In [8]:
import json
import os
from pathlib import Path
from google import genai
from google.genai import types

# ──────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────

INPUT_FILE  = Path("ocr_output_gemini.txt")
OUTPUT_FILE = Path("ner_gemini_output.json") 
MODEL       = "gemma-4-26b-a4b-it"   #gemini-3.1-flash-lite or  gemma-4-26b-a4b-it

# ──────────────────────────────────────────────────────────────────
# SYSTEM PROMPT
# ──────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a bibliographic metadata extractor for an Italian fiscal-history
abstract journal. You will receive a raw text file containing multiple numbered bibliography entries.

Your task is to locate EVERY single bibliography entry in the text, extract its details, 
and return them as a list of JSON objects.

For each entry, extract:
- entry_id: The number of the bibliography entry (e.g., "5882")
- title_it: Italian title as a clean string, no markdown
- title_original: original title in the source language, or null
- author: author name(s) as a string, or null if anonymous
- journal: journal or newspaper name if it is an article, or null
- publisher: publisher name if it is a book, or null
- year: 4-digit year as a string extracted from the date, or null
- place_of_pub: city of publication if explicitly mentioned, or null

Rules:
- Strip all markdown formatting (* and **) from extracted strings.
- For "year": extract only the 4-digit year from dates like "marzo-aprile 1959".
- For "place_of_pub": do NOT infer city from journal names. Only fill if explicitly a place of publication.
- If a field is genuinely absent, return null.
- Maintain initials like "C.L. Harris" as-is.
"""

# ──────────────────────────────────────────────────────────────────
# JSON SCHEMA FOR COMPLIANCE
# ──────────────────────────────────────────────────────────────────

# This forces Gemini to output the exact array format required without needing regex cleanup
entry_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "entry_id": types.Schema(type=types.Type.STRING),
        "title_it": types.Schema(type=types.Type.STRING, nullable=True),
        "title_original": types.Schema(type=types.Type.STRING, nullable=True),
        "author": types.Schema(type=types.Type.STRING, nullable=True),
        "journal": types.Schema(type=types.Type.STRING, nullable=True),
        "publisher": types.Schema(type=types.Type.STRING, nullable=True),
        "year": types.Schema(type=types.Type.STRING, nullable=True),
        "place_of_pub": types.Schema(type=types.Type.STRING, nullable=True),
    },
    required=["entry_id"],
)

response_schema = types.Schema(
    type=types.Type.ARRAY,
    items=entry_schema
)

# ──────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────

def main():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise SystemExit("ERROR: set GOOGLE_API_KEY environment variable first.")

    client = genai.Client(api_key=api_key)

    print(f"Reading {INPUT_FILE}...")
    raw_text = INPUT_FILE.read_text(encoding="utf-8")
    
    print(f"Sending entire text to {MODEL} for processing. Please wait...")

    try:
        response = client.models.generate_content(
            model=MODEL,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0,
                response_mime_type="application/json",
                response_schema=response_schema,
            ),
            contents=raw_text,
        )
        
        # Parse the direct JSON output from the model
        results = json.loads(response.text.strip())
        
        print(f"✓ Successfully processed! Found {len(results)} entries.")
        
        OUTPUT_FILE.write_text(
            json.dumps(results, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        print(f"Results saved to {OUTPUT_FILE}")

    except Exception as e:
        print(f"❌ Critical Error during processing: {e}")


if __name__ == "__main__":
    main()


Reading ocr_output_gemini.txt...
Sending entire text to gemma-4-26b-a4b-it for processing. Please wait...
❌ Critical Error during processing: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'The input token count exceeds the maximum number of tokens allowed (262144).', 'status': 'INVALID_ARGUMENT'}}


Chunked version

In [10]:
import json
import os
import time
from pathlib import Path
from google import genai
from google.genai import types

# ──────────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────────

INPUT_FILE  = Path("ocr_output_gemini.txt")
OUTPUT_FILE = Path("ner_gemini_output.json")
MODEL       = "gemini-3.1-flash-lite"

# ──────────────────────────────────────────────────────────────────
# SYSTEM PROMPT
# ──────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a bibliographic metadata extractor for an Italian fiscal-history
abstract journal. You will receive a raw text file containing multiple numbered bibliography entries.

Your task is to locate EVERY single bibliography entry in the text, extract its details, 
and return them as a list of JSON objects.

For each entry, extract:
- entry_id: The number of the bibliography entry (e.g., "5882")
- title_it: Italian title as a clean string, no markdown
- title_original: original title in the source language, or null
- author: author name(s) as a string, or null if anonymous
- journal: journal or newspaper name if it is an article, or null
- publisher: publisher name if it is a book, or null
- year: 4-digit year as a string extracted from the date, or null
- place_of_pub: city of publication if explicitly mentioned, or null

Rules:
- Strip all markdown formatting (* and **) from extracted strings.
- For "year": extract only the 4-digit year from dates like "marzo-aprile 1959".
- For "place_of_pub": do NOT infer city from journal names. Only fill if explicitly a place of publication.
- If a field is genuinely absent, return null.
- Maintain initials like "C.L. Harris" as-is.
"""

# ──────────────────────────────────────────────────────────────────
# JSON SCHEMA FOR COMPLIANCE
# ──────────────────────────────────────────────────────────────────

entry_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "entry_id": types.Schema(type=types.Type.STRING),
        "title_it": types.Schema(type=types.Type.STRING, nullable=True),
        "title_original": types.Schema(type=types.Type.STRING, nullable=True),
        "author": types.Schema(type=types.Type.STRING, nullable=True),
        "journal": types.Schema(type=types.Type.STRING, nullable=True),
        "publisher": types.Schema(type=types.Type.STRING, nullable=True),
        "year": types.Schema(type=types.Type.STRING, nullable=True),
        "place_of_pub": types.Schema(type=types.Type.STRING, nullable=True),
    },
    required=["entry_id"],
)

response_schema = types.Schema(
    type=types.Type.ARRAY,
    items=entry_schema
)

# ──────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────

def chunk_text(text: str, max_chars: int = 150000) -> list[str]:
    """Splits text into chunks, breaking at double newlines."""
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = []
    current_length = 0
    
    for para in paragraphs:
        if current_length + len(para) + 2 > max_chars and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            current_chunk = [para]
            current_length = len(para)
        else:
            current_chunk.append(para)
            current_length += len(para) + 2
            
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
        
    return chunks


def load_existing_progress() -> list:
    """Loads existing JSON array if the file exists, otherwise returns empty list."""
    if OUTPUT_FILE.exists():
        try:
            return json.loads(OUTPUT_FILE.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            print(f"⚠️ Warning: {OUTPUT_FILE} was corrupted. Starting fresh.")
            return []
    return []

# ──────────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────────

def main():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise SystemExit("ERROR: set GOOGLE_API_KEY environment variable first.")

    client = genai.Client(api_key=api_key)

    print(f"Reading {INPUT_FILE}...")
    raw_text = INPUT_FILE.read_text(encoding="utf-8")
    
    chunks = chunk_text(raw_text, max_chars=150000)
    print(f"File split into {len(chunks)} chunks.")

    # Load any progress made in previous runs
    all_results = load_existing_progress()
    if all_results:
        print(f"⏳ Found existing file with {len(all_results)} entries. Appending new data...")

    MAX_RETRIES = 5
    BASE_DELAY = 4  # Seconds to wait on first failure

    for i, chunk in enumerate(chunks, 1):
        print(f"Processing chunk {i}/{len(chunks)}... ", end="", flush=True)
        
        chunk_success = False
        chunk_results = []

        for attempt in range(MAX_RETRIES):
            try:
                response = client.models.generate_content(
                    model=MODEL,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        temperature=0,
                        response_mime_type="application/json",
                        response_schema=response_schema,
                    ),
                    contents=chunk,
                )
                
                chunk_results = json.loads(response.text.strip())
                chunk_success = True
                break  # Break out of retry loop on success
                
            except Exception as e:
                # If it's a 503 or 429, wait and retry using exponential backoff
                if attempt < MAX_RETRIES - 1:
                    sleep_time = BASE_DELAY * (2 ** attempt)
                    print(f"\n⚠️ Attempt {attempt + 1} failed ({e}). Retrying in {sleep_time}s... ", end="", flush=True)
                    time.sleep(sleep_time)
                else:
                    print(f"\n❌ Critical Error: Chunk {i} failed permanently after {MAX_RETRIES} attempts. Error: {e}")

        if not chunk_success:
            print("Stopping execution. Progress up to this point has been saved.")
            return

        # Successfully got chunk data, update our master list
        all_results.extend(chunk_results)
        print(f"✓ (Added {len(chunk_results)} entries)")

        # Save immediately so we never lose progress
        OUTPUT_FILE.write_text(
            json.dumps(all_results, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        
        # Polite breathing window between chunks
        if i < len(chunks):
            time.sleep(2)

    print(f"\n🎉 All chunks successfully processed! Total entries: {len(all_results)}")


if __name__ == "__main__":
    main()

Reading ocr_output_gemini.txt...
File split into 20 chunks.
Processing chunk 1/20... ✓ (Added 12 entries)
Processing chunk 2/20... ✓ (Added 40 entries)
Processing chunk 3/20... ✓ (Added 31 entries)
Processing chunk 4/20... ✓ (Added 46 entries)
Processing chunk 5/20... ✓ (Added 37 entries)
Processing chunk 6/20... ✓ (Added 46 entries)
Processing chunk 7/20... ✓ (Added 35 entries)
Processing chunk 8/20... ✓ (Added 40 entries)
Processing chunk 9/20... ✓ (Added 48 entries)
Processing chunk 10/20... ✓ (Added 42 entries)
Processing chunk 11/20... ✓ (Added 45 entries)
Processing chunk 12/20... 
⚠️ Attempt 1 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}). Retrying in 4s... ✓ (Added 32 entries)
Processing chunk 13/20... ✓ (Added 41 entries)
Processing chunk 14/20... ✓ (Added 32 entries)
Processing chunk 15/20... ✓ (Added 22 entries)
Process